In [0]:
%sql
-- 1. Ensure Gold Schema exists
CREATE SCHEMA IF NOT EXISTS globalmart.gold;

-- 2. Standard View (Evaluated dynamically on every query)
CREATE OR REPLACE VIEW globalmart.gold.vw_daily_seller_revenue AS
SELECT 
    to_date(cast(d.date_key AS string), 'yyyyMMdd') AS sales_date,
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(f.sales_fact_sk) AS total_items_sold,
    ROUND(SUM(f.total_amount), 2) AS daily_revenue
FROM globalmart.silver.fct_sales f
INNER JOIN globalmart.silver.dim_seller s ON f.seller_sk = s.seller_sk
INNER JOIN globalmart.silver.dim_date d ON f.order_date_key = d.date_key
GROUP BY 
    d.date_key,
    s.seller_id,
    s.seller_state;

-- 3. Persisted Gold Table (Replaces Materialized View)
CREATE OR REPLACE TABLE globalmart.gold.mv_daily_seller_revenue AS
SELECT 
    to_date(cast(d.date_key AS string), 'yyyyMMdd') AS sales_date,
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(f.sales_fact_sk) AS total_items_sold,
    ROUND(SUM(f.total_amount), 2) AS daily_revenue
FROM globalmart.silver.fct_sales f
INNER JOIN globalmart.silver.dim_seller s ON f.seller_sk = s.seller_sk
INNER JOIN globalmart.silver.dim_date d ON f.order_date_key = d.date_key
GROUP BY 
    d.date_key,
    s.seller_id,
    s.seller_state;

In [0]:
%sql
SELECT 
    seller_state,
    COUNT(DISTINCT seller_id) AS active_sellers,
    ROUND(SUM(daily_revenue), 2) AS state_revenue
FROM globalmart.gold.vw_daily_seller_revenue
WHERE seller_state = 'SP' 
  AND sales_date BETWEEN '2018-01-01' AND '2018-12-31'
GROUP BY seller_state;

In [0]:
%sql
SELECT 
    seller_state,
    COUNT(DISTINCT seller_id) AS active_sellers,
    ROUND(SUM(daily_revenue), 2) AS state_revenue
FROM globalmart.gold.mv_daily_seller_revenue
WHERE seller_state = 'SP' 
  AND sales_date BETWEEN '2018-01-01' AND '2018-12-31'
GROUP BY seller_state;

In [0]:
# ==============================================================================
# TASK 6.2 PART A: DAILY SALES SUMMARY TABLE (FIXED)
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Load Delivered Orders Fact Table
df_sales = spark.table("globalmart.silver.fct_sales")

# 2. Window to find each customer's absolute first order date
window_cust = Window.partitionBy("customer_sk")

df_orders_with_cohort = (
    df_sales
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
    .withColumn("first_order_date", F.min("order_date").over(window_cust))
    .withColumn("is_new_customer", F.when(F.col("order_date") == F.col("first_order_date"), 1).otherwise(0))
)

# 3. Aggregate daily metrics (is_late_delivery is treated directly as Boolean)
daily_summary_df = (
    df_orders_with_cohort
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("freight_value"), 2).alias("total_freight"),
        F.countDistinct("customer_sk").alias("total_unique_customers"),
        # New customers
        F.countDistinct(F.when(F.col("is_new_customer") == 1, F.col("customer_sk"))).alias("new_customers"),
        # Returning customers
        F.countDistinct(F.when(F.col("is_new_customer") == 0, F.col("customer_sk"))).alias("returning_customers"),
        F.countDistinct("seller_sk").alias("active_sellers"),
        F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
        # Late delivery percentage calculation
        F.round(F.avg(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)) * 100, 2).alias("late_delivery_pct")
    )
)

# 4. Save to Gold
daily_summary_df.write.format("delta").mode("overwrite").saveAsTable("globalmart.gold.fact_daily_sales_summary")

print(f"✅ Created `globalmart.gold.fact_daily_sales_summary` successfully!")

In [0]:
# ==============================================================================
# TASK 6.2 PART B: MONTHLY SELLER PERFORMANCE TABLE (FIXED)
# ==============================================================================

# 1. Load Facts and Dimensions
df_sales = spark.table("globalmart.silver.fct_sales")
df_seller = spark.table("globalmart.silver.dim_seller")

# 2. Join and aggregate by year-month and seller
df_monthly = (
    df_sales
    .withColumn("year_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .join(df_seller, "seller_sk", "inner")
    .groupBy("year_month", "seller_id", "seller_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("sales_fact_sk").alias("total_items_sold"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
        # Late delivery percentage calculation
        F.round(F.avg(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)) * 100, 2).alias("late_delivery_pct")
    )
)

# 3. Define Window Specifications for Rankings
window_overall = Window.partitionBy("year_month").orderBy(F.col("total_revenue").desc())
window_state = Window.partitionBy("year_month", "seller_state").orderBy(F.col("total_revenue").desc())

# 4. Calculate Overall and State Revenue Rankings
df_seller_perf = (
    df_monthly
    .withColumn("overall_rank", F.dense_rank().over(window_overall))
    .withColumn("state_rank", F.dense_rank().over(window_state))
)

# 5. Save to Gold
df_seller_perf.write.format("delta").mode("overwrite").saveAsTable("globalmart.gold.agg_seller_performance_monthly")

print(f"✅ Created `globalmart.gold.agg_seller_performance_monthly` successfully!")

In [0]:
%sql
-- Validation Check: Print any rows where new + returning != total unique customers
SELECT 
    order_date,
    total_unique_customers,
    new_customers,
    returning_customers,
    (new_customers + returning_customers) AS calculated_total,
    ABS(total_unique_customers - (new_customers + returning_customers)) AS discrepancy
FROM globalmart.gold.fact_daily_sales_summary
WHERE total_unique_customers <> (new_customers + returning_customers)
ORDER BY order_date;

In [0]:
# ==============================================================================
# TASK 6.2 PART B: MONTHLY SELLER PERFORMANCE TABLE (FIXED)
# ==============================================================================

# 1. Load Facts and Dimensions
df_sales = spark.table("globalmart.silver.fct_sales")
df_seller = spark.table("globalmart.silver.dim_seller")

# 2. Join and aggregate by year-month and seller
df_monthly = (
    df_sales
    .withColumn("year_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .join(df_seller, "seller_sk", "inner")
    .groupBy("year_month", "seller_id", "seller_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("sales_fact_sk").alias("total_items_sold"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
        # Late delivery percentage calculation
        F.round(F.avg(F.when(F.col("is_late_delivery") == True, 1).otherwise(0)) * 100, 2).alias("late_delivery_pct")
    )
)

# 3. Define Window Specifications for Rankings
window_overall = Window.partitionBy("year_month").orderBy(F.col("total_revenue").desc())
window_state = Window.partitionBy("year_month", "seller_state").orderBy(F.col("total_revenue").desc())

# 4. Calculate Overall and State Revenue Rankings
df_seller_perf = (
    df_monthly
    .withColumn("overall_rank", F.dense_rank().over(window_overall))
    .withColumn("state_rank", F.dense_rank().over(window_state))
)

# 5. Save to Gold
df_seller_perf.write.format("delta").mode("overwrite").saveAsTable("globalmart.gold.agg_seller_performance_monthly")

print(f"✅ Created `globalmart.gold.agg_seller_performance_monthly` successfully!")


In [0]:
%sql
-- Top 5 Sellers Overall for May 2018
SELECT 
    year_month,
    overall_rank,
    state_rank,
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    avg_delivery_days,
    late_delivery_pct
FROM globalmart.gold.agg_seller_performance_monthly
WHERE year_month = '2018-05'
  AND overall_rank <= 5
ORDER BY overall_rank ASC;

In [0]:
%sql
SELECT 
    COUNT(*) AS total_daily_rows,
    SUM(CASE WHEN total_unique_customers = (new_customers + returning_customers) THEN 1 ELSE 0 END) AS matching_rows,
    SUM(CASE WHEN total_unique_customers <> (new_customers + returning_customers) THEN 1 ELSE 0 END) AS failing_rows
FROM globalmart.gold.fact_daily_sales_summary;

In [0]:
%sql
SELECT 
    year_month,
    MIN(overall_rank) AS min_rank,
    MAX(overall_rank) AS max_rank,
    COUNT(DISTINCT seller_id) AS distinct_sellers
FROM globalmart.gold.agg_seller_performance_monthly
WHERE year_month = '2018-05'
GROUP BY year_month;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
%sql
SHOW TABLES IN globalmart.bronze;

In [0]:
# ==============================================================================
# TASK 6.3 STEP 1: STREAMING SOURCE DIRECTORY SETUP
# ==============================================================================

from pyspark.sql import functions as F

# 1. Set Catalog and Schema context
spark.sql("USE CATALOG globalmart")
spark.sql("USE SCHEMA bronze")

# 2. Create Volume in globalmart.bronze for streaming storage
spark.sql("CREATE VOLUME IF NOT EXISTS globalmart.bronze.streaming_volume")

source_dir = "/Volumes/globalmart/bronze/streaming_volume/orders/"
checkpoint_dir = "/Volumes/globalmart/bronze/streaming_volume/checkpoints_orders/"

# 3. Clean and recreate streaming directories
dbutils.fs.rm(source_dir, recurse=True)
dbutils.fs.rm(checkpoint_dir, recurse=True)
dbutils.fs.mkdirs(source_dir)

# 4. Load baseline bronze orders table (using exact table name: bronze_orders)
df_batch_orders = spark.table("globalmart.bronze.bronze_orders")
batch_count = df_batch_orders.count()

# 5. Stage batch data into Volume as JSON files
df_batch_orders.write.format("json").mode("overwrite").save(source_dir)

print(f"📁 Streaming source prepared at Volume: {source_dir}")
print(f"📊 Initial batch records staged from `bronze_orders`: {batch_count:,}")

In [0]:
# ==============================================================================
# TASK 6.3 STEP 2: AUTO LOADER INGESTION & METADATA COLUMNS (UC COMPATIBLE)
# ==============================================================================

spark.sql("DROP TABLE IF EXISTS globalmart.bronze.orders_stream")

# Read stream via Auto Loader with Unity Catalog metadata syntax
df_orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_dir}/schema")
    .load(source_dir)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Write stream to Delta table
query_initial = (
    df_orders_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)
    .toTable("globalmart.bronze.orders_stream")
)

query_initial.awaitTermination()

stream_count = spark.table("globalmart.bronze.orders_stream").count()

print("✅ Initial streaming execution finished!")
print(f"📊 Batch Table (`bronze_orders`) Count: {batch_count:,}")
print(f"📊 Streaming Table (`orders_stream`) Count: {stream_count:,}")

# Verification check
assert batch_count == stream_count, "⚠️ Discrepancy detected between batch and streaming tables!"
print("🎉 Record count verification PASSED (100% match)!")

In [0]:
# ==============================================================================
# TASK 6.3 STEP 3: SIMULATE NEW ARRIVAL & AUTOMATIC PICKUP (UC COMPATIBLE)
# ==============================================================================

# 1. Create 3 mock incoming order records
new_records = [
    ("ord_stream_99001", "cust_991", "delivered", "2026-07-26 10:00:00", "2026-07-26 10:05:00", "2026-07-26 12:00:00", "2026-07-26 15:00:00", "2026-08-01 00:00:00"),
    ("ord_stream_99002", "cust_992", "delivered", "2026-07-26 11:00:00", "2026-07-26 11:05:00", "2026-07-26 13:00:00", "2026-07-26 16:00:00", "2026-08-01 00:00:00"),
    ("ord_stream_99003", "cust_993", "shipped",   "2026-07-26 12:00:00", "2026-07-26 12:05:00", "2026-07-26 14:00:00", None,                  "2026-08-02 00:00:00")
]

columns = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp", 
    "order_approved_at", "order_delivered_carrier_date", 
    "order_delivered_customer_date", "order_estimated_delivery_date"
]

df_new_arrival = spark.createDataFrame(new_records, columns)

# 2. Save new JSON file to source Volume
df_new_arrival.write.format("json").mode("append").save(source_dir)

# 3. Trigger stream processing
query_incremental = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_dir}/schema")
    .load(source_dir)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)
    .toTable("globalmart.bronze.orders_stream")
)

query_incremental.awaitTermination()

updated_count = spark.table("globalmart.bronze.orders_stream").count()
added_records = updated_count - stream_count

print(f"🚀 Incremental Processing Complete!")
print(f"📊 Previous Count: {stream_count:,}")
print(f"📊 Updated Count:  {updated_count:,} (+{added_records} new records automatically ingested)")

In [0]:
%sql
-- Evidence Query: Display newly ingested records with Auto Loader metadata
SELECT 
    order_id,
    order_status,
    _ingested_at,
    _source_file
FROM globalmart.bronze.orders_stream
WHERE order_id LIKE 'ord_stream_%'
ORDER BY _ingested_at DESC;

In [0]:
%sql
-- 1. Create permissions table
CREATE TABLE IF NOT EXISTS globalmart.bronze.user_state_permissions (
    user_email STRING,
    state STRING
);

-- 2. Insert test user state mapping (maps current session user to SP and RJ states)
INSERT INTO globalmart.bronze.user_state_permissions 
VALUES 
    (CURRENT_USER(), 'SP'),
    (CURRENT_USER(), 'RJ');

In [0]:
%sql
-- Dynamic Column Masking View
CREATE OR REPLACE VIEW globalmart.bronze.v_secure_customers AS
SELECT 
    customer_id,
    customer_unique_id,
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN customer_zip_code_prefix
        ELSE 'XXXXX'
    END AS customer_zip_code_prefix,
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN customer_city
        ELSE 'REDACTED'
    END AS customer_city,
    customer_state
FROM globalmart.bronze.bronze_customers;

In [0]:
%sql
-- Insert test user state mapping (maps current session user to SP and RJ states)
INSERT INTO globalmart.bronze.user_state_permissions (user_email, allowed_state)
VALUES 
    (CURRENT_USER(), 'SP'),
    (CURRENT_USER(), 'RJ');

In [0]:
%sql
-- TASK 6.4 PART B: ROW-LEVEL SECURITY VIEW
CREATE OR REPLACE VIEW globalmart.bronze.v_secure_orders AS
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    c.customer_state
FROM globalmart.bronze.bronze_orders o
INNER JOIN globalmart.bronze.bronze_customers c 
    ON o.customer_id = c.customer_id
WHERE 
    IS_ACCOUNT_GROUP_MEMBER('admin') 
    OR c.customer_state IN (
        SELECT allowed_state 
        FROM globalmart.bronze.user_state_permissions 
        WHERE user_email = CURRENT_USER()
    );

In [0]:
%sql
-- TASK 6.4 PART A: DYNAMIC COLUMN MASKING VIEW (CAST FIX)
CREATE OR REPLACE VIEW globalmart.bronze.v_secure_customers AS
SELECT 
    customer_id,
    customer_unique_id,
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN CAST(customer_zip_code_prefix AS STRING)
        ELSE 'XXXXX'
    END AS customer_zip_code_prefix,
    CASE 
        WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN customer_city
        ELSE 'REDACTED'
    END AS customer_city,
    customer_state
FROM globalmart.bronze.bronze_customers;

In [0]:
%sql
SELECT 
    customer_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM globalmart.bronze.v_secure_customers
LIMIT 5;

In [0]:
%sql
SELECT 
    customer_state,
    COUNT(order_id) AS total_visible_orders
FROM globalmart.bronze.v_secure_orders
GROUP BY customer_state
ORDER BY total_visible_orders DESC;